In [1]:
#importing the Libraies
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv("CKD.csv")

In [3]:
dataset

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [4]:
dataset.columns

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')

In [5]:
indep=dataset[['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane']]
dep=dataset[['classification']]

In [6]:

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

#Navie bayes algorithm dosn't predict scalled dataset so we drop standardscaller 

In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_en = encoder.fit_transform(X_train)
X_test_en = encoder.transform(X_test)

In [8]:
dataset["classification"].value_counts()

classification
yes    249
no     150
Name: count, dtype: int64

In [11]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
param_grid ={
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}

grid = GridSearchCV(GaussianNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
grid.fit(X_train_en, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


C:\Anaconda3.12.v\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(estimator=GaussianNB(), n_jobs=-1,
             param_grid={'var_smoothing': [1e-09, 1e-08, 1e-07, 1e-06]},
             scoring='f1_weighted', verbose=3)

In [12]:
print(grid.best_params_)

{'var_smoothing': 1e-09}


In [14]:
y_pred = grid.predict(X_test_en)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred, target_names=["CKD_GNB_NO","CKD_GNB_YES"])

In [15]:
print(cm)
print(clf_report)

[[47  4]
 [ 2 80]]
              precision    recall  f1-score   support

  CKD_GNB_NO       0.96      0.92      0.94        51
 CKD_GNB_YES       0.95      0.98      0.96        82

    accuracy                           0.95       133
   macro avg       0.96      0.95      0.95       133
weighted avg       0.95      0.95      0.95       133



In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
param_grid ={'alpha': [0.1, 0.5, 1.0], 'fit_prior': [True, False]}

grid = GridSearchCV(MultinomialNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   

grid.fit(X_train_en, y_train) 
 

Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Anaconda3.12.v\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(estimator=MultinomialNB(), n_jobs=-1,
             param_grid={'alpha': [0.1, 0.5, 1.0], 'fit_prior': [True, False]},
             scoring='f1_weighted', verbose=3)

In [17]:
print(grid.best_params_)

{'alpha': 0.1, 'fit_prior': True}


In [18]:

y_pred = grid.predict(X_test_en)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred, target_names=["CKD_MNB_NO","CKD_MNB_YES"])              


In [19]:
print(cm)
print(clf_report)


[[51  0]
 [ 1 81]]
              precision    recall  f1-score   support

  CKD_MNB_NO       0.98      1.00      0.99        51
 CKD_MNB_YES       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



In [20]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test_en)[:,1])

0.998804399808704

In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import BernoulliNB
param_grid = param_grid = {'alpha': [0.1, 0.5, 1.0],
    'binarize': [0.0, 0.5, 1.0],
    'fit_prior': [True, False]}
grid = GridSearchCV(BernoulliNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train_en, y_train) 

Fitting 5 folds for each of 18 candidates, totalling 90 fits


C:\Anaconda3.12.v\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(estimator=BernoulliNB(), n_jobs=-1,
             param_grid={'alpha': [0.1, 0.5, 1.0], 'binarize': [0.0, 0.5, 1.0],
                         'fit_prior': [True, False]},
             scoring='f1_weighted', verbose=3)

In [23]:
print(grid.best_params_)

{'alpha': 0.5, 'binarize': 0.0, 'fit_prior': True}


In [24]:
y_pred = grid.predict(X_test_en)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred, target_names=["CKD_BNB_NO","CKD_BNB_YES"])             


In [25]:
print(cm)
print(clf_report)


[[51  0]
 [ 2 80]]
              precision    recall  f1-score   support

  CKD_BNB_NO       0.96      1.00      0.98        51
 CKD_BNB_YES       1.00      0.98      0.99        82

    accuracy                           0.98       133
   macro avg       0.98      0.99      0.98       133
weighted avg       0.99      0.98      0.99       133



In [27]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test_en)[:,1])

0.9992826398852224

In [28]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import CategoricalNB
param_grid = param_grid = {'alpha': [0.1, 0.5, 1.0], 'fit_prior': [True, False]}
grid = GridSearchCV(CategoricalNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train_en, y_train) 

Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Anaconda3.12.v\Lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
  warnings.warn(
C:\Anaconda3.12.v\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(estimator=CategoricalNB(), n_jobs=-1,
             param_grid={'alpha': [0.1, 0.5, 1.0], 'fit_prior': [True, False]},
             scoring='f1_weighted', verbose=3)

In [29]:
print(grid.best_params_)

{'alpha': 0.1, 'fit_prior': True}


In [30]:
y_pred = grid.predict(X_test_en)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred, target_names=["CKD_CATENB_NO","CKD_CATENB_YES"])             
              


In [31]:
print(cm)
print(clf_report)

[[51  0]
 [ 1 81]]
                precision    recall  f1-score   support

 CKD_CATENB_NO       0.98      1.00      0.99        51
CKD_CATENB_YES       1.00      0.99      0.99        82

      accuracy                           0.99       133
     macro avg       0.99      0.99      0.99       133
  weighted avg       0.99      0.99      0.99       133



In [32]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test_en)[:,1])

0.9990435198469633

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import  ComplementNB
param_grid = param_grid = {
    'alpha': [0.1, 0.5, 1.0, 2.0],
    'norm': [True, False]}

grid = GridSearchCV( ComplementNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train_en, y_train) 

Fitting 5 folds for each of 8 candidates, totalling 40 fits


C:\Anaconda3.12.v\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(estimator=ComplementNB(), n_jobs=-1,
             param_grid={'alpha': [0.1, 0.5, 1.0, 2.0], 'norm': [True, False]},
             scoring='f1_weighted', verbose=3)

In [34]:
print(grid.best_params_)

{'alpha': 0.1, 'norm': False}


In [35]:
y_pred = grid.predict(X_test_en)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, y_pred, target_names=["CKD_COMNB_NO","CKD_COMNB_YES"])           


In [36]:
print(cm)
print(clf_report)

[[51  0]
 [ 1 81]]
               precision    recall  f1-score   support

 CKD_COMNB_NO       0.98      1.00      0.99        51
CKD_COMNB_YES       1.00      0.99      0.99        82

     accuracy                           0.99       133
    macro avg       0.99      0.99      0.99       133
 weighted avg       0.99      0.99      0.99       133



In [37]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test_en)[:,1])

0.998804399808704